# App Platform Contract — Design Spec

**Date:** 2026-06-10 · **Design epic:** `bd-bbig` · **Status:** approved design, pending implementation plan
**Companion spec:** `2026-06-10-spur-app-sdk-design.ipynb` (depends on this spec)

Formalizes the contract between a Spur app bundle and the host notebook: capability declaration → host provisioning → verification. Host-side (Rust) work only; the app-side SDK is the companion spec.

## 1. Problem & evidence

The app model has strong identity/lifecycle/packaging machinery but **no formal capability layer**. Every defect found in the html_video review (2026-06-10) is a violation at this unowned seam:

| Defect | Evidence | Missing primitive |
|---|---|---|
| `SPUR_PORTS_ROOT` never injected; `port_names` render always fails | `plugin_loader.rs:123` passes only manifest env + `PYTHONUNBUFFERED`; html-video spec line 72 required foundation injection — never implemented | Host-provisioned env |
| Python server reads `root/<port>` — file is `<port>@v<N>.media` | `render.py:369` vs `ports.rs:294-301`; prose contract in spec §Port Access diverged from code | Machine-checkable store contract |
| Capture loop half-built: skill promises `notebook_get_cell_capture`, tool doesn't exist | spec line 303: "separate design"; real path is `push_capture_port` → port store | Capability with a complete producer→consumer loop |
| Output scripts blocked by default → capture silently never runs | `settings.ts:29` `activeContent: false`, global only | Per-app trust grant |
| 60s capture renders as 3s mp4 | `commands.rs:210` validates `duration_sec`, `commands.rs:227` drops it; `render.py` defaults `frame_duration=3.0` | Media metadata in store contract |
| `runtime.features` looks like capabilities, enforces nothing | `jute_min`/`features` written, never read for gating | Enforcement |
| App skill is convention-only (`skill/SKILL.md`), packer unaware | no `skill` field in `SpurAppManifest`; `spur_app.rs` has no reference | Skill as declared contract element |

## 2. Goals / non-goals

**Goals**
1. Additive `capabilities` field on `spur.app/v1` with enforced semantics: declare → provision → verify.
2. Host provisioning at app-plugin spawn, following the existing kernel-env injection pattern.
3. Port-store wire contract as shared golden fixtures (Rust writer pins; SDK readers pin in companion spec) + `duration_sec` on media entries.
4. Per-app trust grant for active output scripts, replacing the silent global default.
5. `notebook_app_doctor` MCP tool: conformance checks runnable at open and in CI.
6. html_video Phase-1 fixes as the contract's first consumer.

**Non-goals**
- App-side SDKs, CLI front doors, scaffolding, publishing (companion spec).
- New capture infrastructure (`notebook_get_cell_capture` stays out of scope; the port-based loop is the contract).
- Capability sandboxing/enforcement beyond grant-or-refuse (no syscall-level isolation in v1).
- Remote/registry app distribution.

## 3. Capability contract (manifest extension)

Additive serde field on `SpurAppManifest` (`crates/spur-notebook/src/spur_app.rs:27`); existing manifests deserialize unchanged. Also additive: `skill` (path to the app's agent skill, default `"skill/SKILL.md"`).

```json
{
  "schema": "spur.app/v1",
  "capabilities": {
    "ports": { "read": ["spur-ad-capture"], "write": [] },
    "canvas_capture": true,
    "active_output_scripts": true,
    "artifacts_dir": true
  },
  "skill": "skill/SKILL.md"
}
```

| Capability | Host semantics |
|---|---|
| `ports` | Inject `SPUR_PORTS_ROOT` at plugin spawn (exact value §4). Doctor verifies declared `read` ports are declared as DAG sources in the entry notebook. |
| `canvas_capture` | Host guarantees the recorder loop end-to-end: `withVideoCapture` injection (`rendering.ts:9`) → `jute-video-capture` postMessage → `push_capture_port` (`commands.rs:183`) → port store, incl. `duration_sec` persistence (§5). Requires `active_output_scripts`. |
| `active_output_scripts` | App-mode open shows a one-time per-app grant prompt (§6); granted apps get `allow-scripts allow-same-origin` output iframes regardless of the global toggle. |
| `artifacts_dir` | Inject `SPUR_ARTIFACTS_DIR` (host-owned dir under the notebook runtime root) at plugin spawn and into kernel env; replaces `Deno.cwd()` output-path guessing. |

**Enforcement:** unknown capability key, or a capability the host cannot provision → app open is refused with a structured diagnostic naming the key and the manifest path. `runtime.features` is deprecated but honored (ignored with a doctor warning). This is the enforcement `jute_min`/`features` never had.

## 4. Host provisioning at plugin spawn

**Prior art (verified):** the host already injects `SPUR_NOTEBOOK_PORT_ROOT` and `SPUR_NOTEBOOK_MCP_SOCKET` into *kernel* environments — `src-tauri/src/commands.rs:51-52`, `apply_notebook_port_root_env` / `apply_notebook_mcp_socket_env` (`commands.rs:1170-1190`, callsites `1222`, `2345`). This spec applies the same pattern to the app-plugin spawn path.

**Change site:** `app_plugin_config_for_notebook` / `spawn_app_plugin` (`crates/spur-notebook/src/mcp/mod.rs:3001-3070`) — the notebook path is already in hand. Provisioned env is merged into `PluginConfig` env *after* manifest env (host wins on conflict, with a spawn-time warning).

**Exact values (verified against `PortStore::open_at`, `ports.rs:223` — it joins `"ports"` internally):**
- `SPUR_PORTS_ROOT` = `notebook_port_root(entry_notebook_path)/ports` — the directory containing `manifest.json` and the versioned port files. `notebook_port_root` resolves to `~/.spur/notebooks/<blake3-id>` (`jute-notebook/src-tauri/src/ports.rs:18`).
- `SPUR_ARTIFACTS_DIR` = `notebook_port_root(entry_notebook_path)/artifacts` (host creates it).

Injection is driven by the `capabilities` declarations — no declaration, no injection.

## 5. Port-store contract: golden fixtures + `duration_sec`

**Fixtures.** Extract the existing `js_manifest_shape_deserializes_into_port_manifest` test data (`ports.rs:561`) into `crates/spur-notebook/fixtures/port-store/` — a manifest.json + sample `@v1.arrow` / `@v1.media` files. Rust round-trip tests pin against the fixture files; the companion-spec SDKs ship and pin the same files. A wire-format change that updates one side without the other fails CI on both.

**Contract facts the fixtures encode (all verified):** top-level `{"ports": {<name>: entry}}`; entry = `{path, version, kind, mime?, size?, schema?, duration_sec?}`; `path` is the absolute versioned file (`<port>@v<N>.<ext>`); consumers MUST read `entry.path`, never derive `root/<port>`.

**`duration_sec` (additive, media entries only).** Threads through exactly four sites:
1. `commands.rs:227` — `push_capture_port_for_state` forwards the already-validated value;
2. `engine.rs:64` — `SourcePayload::MediaBlob` gains `duration_sec: Option<f64>`;
3. `engine.rs:332` and `engine.rs:658` — the two `SourcePayload→PortPayload` mappings;
4. `ports.rs` — `PortPayload::MediaBlob`, `PortKind::Media`, and the wire entry serialize/deserialize (optional field; old manifests default `None`).

## 6. Per-app trust grant

The settings store already persists via zustand `persist`/localStorage (`settings.ts:42-65`). Extend it with `appGrants: Record<appRootPath, { activeOutputScripts: boolean, grantedAt: string }>`.

**Flow:** opening a notebook whose manifest declares `active_output_scripts` and has no stored grant → modal listing the app name + requested capabilities → Allow persists the grant; Deny opens the app with scripts off and a visible banner (not silent). `HtmlOutput` (`OutputView.tsx:244`) resolves trust as `appGrant ?? globalActiveContent`. Revocation lives in the existing settings panel.

**Error handling rule (uniform across this spec):** every contract violation is a structured diagnostic — open-refusal dialog, doctor finding, or spawn warning — naming the violated contract element and the manifest path. Never a silent default.

## 7. `notebook_app_doctor` (host-side conformance core)

New MCP tool in the notebook daemon (next to `notebook_export_spur_app`). Input: app root path. Output: structured findings `{check, level: pass|warn|fail, message, location}`.

**Checks (v1):**
1. Manifest parses; schema is `spur.app/v1`; `entry_notebook` exists.
2. Every declared capability is known and grantable on this host.
3. Declared `ports.read` names exist as DAG sources (kind `canvas-capture` or producer cells) in the entry notebook.
4. Plugin spawns; `tools/list` succeeds (reuses the lifecycle machinery proven by `mcp/mod.rs:5838` test).
5. Skill file at manifest `skill` path exists; every tool name referenced in its HARD-GATE block appears in the live tool surface (plugin tools + notebook MCP tools). Catches phantom-tool drift mechanically.
6. Port store reachable at the provisioned `SPUR_PORTS_ROOT`; fixtures version compatible.
7. `runtime.features` present → warn deprecated.

CLI/agent front doors ship in the companion spec; this spec delivers the tool and its checks.

## 8. html_video Phase-1 fixes (first consumer, folded in)

1. `app_gallery/html_video/server/render.py` — `read_webm_port_frames` reads `entry["path"]` from the manifest (basename-joined under root for safety) instead of `root/<port>`; honors `entry["duration_sec"]` as default frame duration when `frame_duration` is not passed.
2. `app.ipynb` `spur-ad-render` cell — passes `frame_duration: 60` until (1)+§5 land end-to-end.
3. `spur-app.json` — declares `capabilities` (ports.read `spur-ad-capture`, `canvas_capture`, `active_output_scripts`, `artifacts_dir`) and `skill`.
4. `skill/SKILL.md` + `references/video-mode.md` — rewritten to the port-based flow: drop `notebook_get_cell_capture` and the unreachable `webm_frames` recipe; document `port_names`, the trust grant, and the real 5-template `index.json` catalog.

TDD cadence per repo convention: failing `test(...)` commit first for each fix.

## 9. Task decomposition boundaries & acceptance

**DAG (each node ≈ one plan task):**
- T1 `capabilities` + `skill` manifest schema (spur_app.rs) — root.
- T2 spawn provisioning (mcp/mod.rs) — depends T1.
- T3 port-store fixtures + `duration_sec` (ports.rs, engine.rs, commands.rs) — independent of T1/T2.
- T4 per-app trust grant (frontend settings + OutputView + open prompt) — depends T1 (reads manifest), parallel with T2/T3.
- T5 doctor tool (daemon) — depends T1, T2; check 6 depends T3.
- T6 html_video Phase-1 fixes — render.py/notebook parts parallel-safe now; manifest/skill parts depend T1.

**Acceptance criteria:**
- Fresh checkout → open `app_gallery/html_video/app.ipynb` in app mode → single grant prompt → capture (60s) → render → embedded MP4 of ~60s, zero manual env setup.
- `notebook_app_doctor` green on html_video in CI; red if a HARD-GATE tool name is removed from the tool surface (regression test for skill drift).
- Existing manifests without `capabilities` open exactly as today (additive-compat test).
- Round-trip serialization tests for all new manifest/wire fields, modeled on `executor_events_roundtrip.rs` per repo convention.

## 10. References

- Review findings & double-confirmation: session 2026-06-10 (epic `bd-bbig` comments).
- `docs/superpowers/specs/2026-06-08-html-video-app-gallery-design.md` — prior spec whose §Env-var-injection (line 72) and §Port Access (lines 195-204) this spec implements/corrects.
- `docs/superpowers/specs/2026-06-09-open-mode-viewmode-propagation-design.md` — presentation seam (done).
- Key symbols: `SpurAppManifest` (`spur_app.rs:27`), `app_plugin_config_for_notebook`/`spawn_app_plugin` (`mcp/mod.rs:3001-3070`), `PluginConfig::command_env` (`plugin_loader.rs:123`), `PortStore::open_at` (`ports.rs:223`), `PortStore::put` (`ports.rs:261`), `push_capture_port` (`commands.rs:183`), `SourcePayload` (`engine.rs:62`), `notebook_port_root` (`src-tauri/ports.rs:18`), `withVideoCapture` (`rendering.ts:9`), `HtmlOutput` (`OutputView.tsx:244`), settings persist (`settings.ts:42-65`).
- Graph state at grounding: head `2e5d340a6`, `response_file_oids_match: true`.